### Medical ChatBot - Langchain, Pinecone, OpenAI

In [1]:
#import the langchain libraries

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/apple/Desktop/Langchain/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Extract the Data from the PDF File

def load_pdf(data):
    loader = DirectoryLoader(data,glob="*.pdf",loader_cls=PyPDFLoader)
    documents = loader.load()
    
    return documents
    

In [3]:
extracted_data = load_pdf(data='data/')

In [4]:
#Split the data into text chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [5]:
text_chunks=text_split(extracted_data)
print("Length of the text chunks",len(text_chunks))

Length of the text chunks 5859


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
# After splitting the extracted data into chunks download an embedding model from Hugging Face
#model_name = "\sentence-transformers"
def download_huggingface_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return embeddings

In [10]:
embeddings = download_huggingface_embeddings()

In [11]:
query=embeddings.embed_query("hello this is manish")
print(len(query))

384


In [ ]:
pip install pinecone

In [15]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name="medicalchatbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws",
                        region="us-east-1")
)






{
    "name": "medicalchatbot",
    "metric": "cosine",
    "host": "medicalchatbot-tjisgco.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [16]:
#loading my api key once again
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

### Sroring the vector embeddings in the Vector store

In [ ]:
#pip install langchain-pinecone


In [17]:
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [18]:
#we can also load the existing index as well
existing_index = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [19]:
existing_index

In [22]:
retriver = existing_index.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [23]:
retrived_docs = retriver.invoke("What is Acne?")

### Initialize the Model

In [25]:
OPEN_API_KEY=os.environ.get('OPENAI_API_KEY')

In [32]:
from langchain_openai import OpenAI
model_llm=OpenAI(temperature=0.4,max_tokens=500)

In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [29]:
retriver = existing_index.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [27]:
system_prompt = (
    "You are an assistant for question-answering tasks"
    "Use the following pieces of retrived context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know the answer. Use three sentences maximum and keep the answer concise."
    "\n \n"
    "{context}"
)

In [28]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [33]:
rag_chain = (
    {
        "context": retriver,
        "input": RunnablePassthrough()
    }
    | prompt
    | model_llm
    | StrOutputParser() )

In [38]:
response = rag_chain.invoke("What are the symptoms of Acne? And what are the medicines I can use?")
print(response)




Acne is a skin condition that causes pimples, blackheads, and whiteheads. Some common symptoms include oily skin, redness, and inflammation. As for medicines, topical treatments like benzoyl peroxide and salicylic acid are often used, as well as oral medications like antibiotics and hormonal treatments. It is important to consult with a doctor or dermatologist to determine the best course of treatment for your specific case.


In [39]:
response2 = rag_chain.invoke("How Heart attacks are causing?")
print(response2)


A heart attack can occur when there is a complete blockage of blood flow in the coronary arteries. It can also cause a stroke if the brain's arteries are completely blocked. Atherosclerosis, or the buildup of plaque in the arteries, can also lead to heart attacks.
